In [1]:
import torch
from mobilenetv3 import MobileNetV3Large
from ssd import SSDLite

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nb_classes = 21  # VOC: must match checkpoint (same as training `nb_classes`)


# Same as SsdTrainingPipelineVOC2007 when --optimizer adam (PyTorch default betas/eps)

from torchvision.models import MobileNet_V3_Small_Weights
from mobilenetv3 import MobileNetV3Small
weights = MobileNet_V3_Small_Weights.DEFAULT
state_dict = weights.get_state_dict()
model = MobileNetV3Small(0.1, 1000, 1024)
new_state_dict = {}
for my_key, pretrained_key in zip(model.state_dict().keys(), state_dict.keys()):
    new_state_dict[my_key] = state_dict[pretrained_key]
model.load_state_dict(new_state_dict)
backbone=model.features[:-1]

model = SSDLite(
    backbone_config_path="config/ssdlite_mobilenetv3small.yaml",
    backbone=backbone,
    c4_name="6.features.1.features.2",
    nb_classes=nb_classes,
    phase="test",
    alpha=1.0,
    prob_thr=0.01,
    nms_thr=0.45,
    top_k=200,
    variances=[0.1, 0.2],
    N_epochs=50,
    device=device,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0005,
)

car=0
for param in model.parameters():
    car=car+param.numel()
print(car)


import time 
def measure_fps_cpu(model, device=torch.device("cpu"), input_size=300, n_warmup=10, n_iters=100):
    model.eval()
    model.phase = "test"
    model.to(device)

    x = torch.randn(1, 3, input_size, input_size, device=device)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)

        t0 = time.perf_counter()
        for _ in range(n_iters):
            _ = model(x)
        elapsed = time.perf_counter() - t0

    fps = n_iters / elapsed
    print(f"~{fps:.2f} FPS ({n_iters} iters, {input_size}x{input_size}, {device})")
    return fps

# usage after model is loaded
measure_fps_cpu(model)


1665212
~1.96 FPS (100 iters, 300x300, cpu)


1.9580411680353442

MobileNetV3Large

In [ ]:
import torch
from mobilenetv3 import MobileNetV3Large
from ssd import SSDLite

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nb_classes = 21  # VOC: must match checkpoint (same as training `nb_classes`)


# Same as SsdTrainingPipelineVOC2007 when --optimizer adam (PyTorch default betas/eps)



state_dict = weights.get_state_dict()
model = MobileNetV3Large(0.1, 1000, 1280)


backbone=model.features[:-1]

model = SSDLite(
    backbone_config_path="config/ssdlite_mobilenetv3large.yaml",
    backbone=backbone,
    c4_name="7.features.1.features.2",
    nb_classes=nb_classes,
    phase="test",
    alpha=1.0,
    prob_thr=0.01,
    nms_thr=0.45,
    top_k=200,
    variances=[0.1, 0.2],
    N_epochs=50,
    device=device,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0005,
)
from train import load_model
# Weights: train with backbone mobilenetv3large and nb_classes=21
CHECKPOINT = "ssd_voc2007_mv3large_aug.pth"
model , ep, optimizer,maxmap,wandbid=load_model(CHECKPOINT, device, model, optimizer)
car=0
for param in model.parameters():
    car=car+param.numel()
print(car)
import time 
def measure_fps_cpu(model, device=torch.device("cpu"), input_size=300, n_warmup=10, n_iters=100):
    model.eval()
    model.phase = "test"
    model.to(device)

    x = torch.randn(1, 3, input_size, input_size, device=device)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(x)

        t0 = time.perf_counter()
        for _ in range(n_iters):
            _ = model(x)
        elapsed = time.perf_counter() - t0

    fps = n_iters / elapsed
    print(f"~{fps:.2f} FPS ({n_iters} iters, {input_size}x{input_size}, {device})")
    return fps

# usage after model is loaded
measure_fps_cpu(model)

3855308
~3.36 FPS (100 iters, 300x300, cpu)


3.3555861173226593